In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

In [2]:
# ==========================================
# Your existing data
# X 5D (20,5)
# y 1D (20,)
# ==========================================

# Example
X = np.load('function_6/initial_inputs.npy')
y = np.load('function_6/initial_outputs.npy')

X = np.vstack([X, [0.466600, 0.240278, 0.576273, 0.961656, 0.004910],
               [0.549860, 0.322433, 0.619992, 0.914793, 0.158972],
               	[0.465759, 0.196477, 0.978806, 0.961656, 0.301484],
                [0.021735, 0.455929, 0.480914, 0.961656, 0.280518],
                [0.538266, 0.416181, 0.454942, 0.961656, 0.123072],
                [0.496264, 0.315090, 0.611010, 0.744767, 0.147780],
                [0.439712, 0.352071, 0.561400, 0.654564, 0.016737],
                [0.441703, 0.211579, 0.546466, 0.715568, 0.187814],
                [0.506585, 0.350491, 0.667515, 0.739901, 0.004910],
                [0.491376, 0.385002, 0.659356, 0.709950, 0.172084],
                [0.551673, 0.327446, 0.568129, 0.725668, 0.007830]])

y = np.append(y, [-0.5611934994392782, 
                  -0.4274062094396645,
                  -1.0570354342295885,
                  -1.0823058386224655,
                  -0.6364402090985678,
                  -0.15245492174840458,
                  -0.26027100539217335,
                  -0.3599142303595712,
                  -0.2003695065769609,
                  -0.23500859747748837,
                  -0.38567363732658805])

In [3]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=np.ones(X.shape[1]), nu=2.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# ==========================================
# Expected Improvement
# ==========================================

def expected_improvement(x, gp, y_best, xi=0.001):
    x = np.array(x).reshape(1, -1)

    mu, sigma = gp.predict(x, return_std=True)

    sigma = sigma[0]
    mu = mu[0]

    if sigma < 1e-12:
        return 0.0

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei



# ==========================================
# Optimise acquisition function
# ==========================================

bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits

y_best = np.max(y)


def objective(x):
    return -expected_improvement(x, gp, y_best)


best_x = None
best_ei = -np.inf

for _ in range(5000):

    x0 = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds]
    )

    res = minimize(
        objective,
        x0=x0,
        bounds=bounds,
        method="L-BFGS-B"
    )

    ei = -res.fun

    if ei > best_ei:
        best_ei = ei
        best_x = res.x

print("Next point to evaluate:")
next_point = np.round(best_x, decimals=6)
print(next_point)

print("Expected Improvement:")
print(best_ei)

Next point to evaluate:
[0.430832 0.379172 0.684445 0.694663 0.046994]
Expected Improvement:
0.012253265810310561
